# LeetCode 1419 — Minimum Number of Frogs Croaking bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-frogs-croaking/ · Độ khó: **Medium** · Chủ đề: String, Counting (thêm: Simulation, Greedy)

Skeleton LeetCode:

```python
class Solution:
    def minNumberOfFrogs(self, croakOfFrogs: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hoá bài bằng **máy trạng thái**: mỗi con ếch đang ở một trong các trạng thái "vừa kêu `c` / `r` / `o` / `a` / `k`".
2. Hiểu chiến lược **tham lam**: một ký tự luôn có thể giao cho *bất kỳ* con ếch nào đang chờ đúng ký tự đó; chỉ tạo ếch mới khi không có ếch rảnh.
3. Viết lời giải `O(n)` thời gian, `O(1)` bộ nhớ chỉ với **5 bộ đếm**, và phiên bản "số ếch đang kêu = #c − #k".
4. Nhìn bài dưới góc độ **khoảng thời gian chồng nhau** (mỗi tiếng `croak` là một đoạn `[c, k]`), đáp án = số đoạn chồng nhau tối đa.
5. Không quên kiểm tra **tính hợp lệ**: sai thứ tự ở tiền tố, hoặc còn tiếng kêu dở dang ở cuối ⇒ `-1`.

## 2. Đề bài & ví dụ

Cho chuỗi `croakOfFrogs` là sự **trộn lẫn** của nhiều tiếng `"croak"` từ các con ếch khác nhau (nhiều ếch có thể kêu cùng lúc, nên các chữ cái xen kẽ nhau).

Một tiếng kêu hợp lệ: con ếch phát ra lần lượt 5 chữ `'c'`, `'r'`, `'o'`, `'a'`, `'k'`. Ếch phải kêu **đủ cả 5 chữ** mới xong một tiếng; xong rồi nó có thể kêu tiếp tiếng mới.

Trả về **số ếch ít nhất** để tạo ra chuỗi đã cho. Nếu chuỗi **không** phải là sự trộn lẫn của các tiếng `"croak"` hợp lệ, trả `-1`.

**Ví dụ 1:** `"croakcroak"` → `1` (một con ếch kêu hai lần).

**Ví dụ 2:** `"crcoakroak"` → `2` (ếch 1 kêu các chữ ở vị trí 0,1,3,4,5; ếch 2 kêu các chữ ở vị trí 2,6,7,8,9).

**Ví dụ 3:** `"croakcrook"` → `-1`.

**Ràng buộc:** `1 <= croakOfFrogs.length <= 10^5`; mọi ký tự thuộc `{'c','r','o','a','k'}`.

In [1]:
import random
import time
from collections import Counter
from functools import lru_cache
from typing import List

examples = [("croakcroak", 1), ("crcoakroak", 2), ("croakcrook", -1)]
for s, want in examples:
    print(f"{s!r:15} -> {want}")

'croakcroak'    -> 1
'crcoakroak'    -> 2
'croakcrook'    -> -1


## 3. Kiến thức nền

### 3.1 Ánh xạ ký tự → chỉ số bằng `dict` / `str.index`

Ta đánh số `c=0, r=1, o=2, a=3, k=4`. Tạo sẵn `IDX = {ch: i for i, ch in enumerate("croak")}` để tra `O(1)` (gọi `"croak".index(ch)` mỗi lần cũng được vì chuỗi chỉ dài 5).

In [2]:
IDX = {ch: i for i, ch in enumerate("croak")}
print(IDX, "croak".index("a"))
print([IDX[ch] for ch in "crcoakroak"])

{'c': 0, 'r': 1, 'o': 2, 'a': 3, 'k': 4} 3
[0, 1, 0, 2, 3, 4, 1, 2, 3, 4]


### 3.2 Mảng đếm (counting)

`cnt = [0] * 5` — `cnt[i]` là số ếch **vừa** kêu xong chữ thứ `i`, đang chờ kêu chữ `i+1`. Chuyển một ếch từ trạng thái `i-1` sang `i`: `cnt[i-1] -= 1; cnt[i] += 1`.

In [3]:
cnt = [0] * 5
cnt[0] += 1            # một ếch kêu 'c'
cnt[0] -= 1; cnt[1] += 1   # ếch đó kêu 'r'
print(cnt)

[0, 1, 0, 0, 0]


### 3.3 `max` cập nhật dần và `Counter`

Đáp án là **giá trị lớn nhất** của một đại lượng thay đổi theo thời gian (số ếch đang kêu) ⇒ cập nhật `best = max(best, cur)` trong vòng lặp. `Counter(s)` đếm số lần mỗi chữ — dùng để kiểm tra nhanh điều kiện cần (5 chữ phải xuất hiện bằng nhau).

In [4]:
cur, best = 0, 0
for delta in [+1, +1, -1, +1, -1, -1]:
    cur += delta
    best = max(best, cur)
print("best =", best)
print(Counter("crcoakroak"))

best = 2
Counter({'c': 2, 'r': 2, 'o': 2, 'a': 2, 'k': 2})


### 3.4 Quét sự kiện (sweep line)

Có nhiều đoạn `[l, r]`; số đoạn chồng nhau nhiều nhất = giá trị lớn nhất của tổng `+1` tại điểm mở và `-1` sau điểm đóng khi quét theo thời gian.

In [5]:
segs = [(0, 5), (2, 9), (6, 10)]
events = sorted([(l, +1) for l, r in segs] + [(r + 1, -1) for l, r in segs])
cur = best = 0
for t, d in events:
    cur += d
    best = max(best, cur)
print(events, "-> chồng nhau tối đa", best)

[(0, 1), (2, 1), (6, -1), (6, 1), (10, -1), (11, -1)] -> chồng nhau tối đa 2


## 4. Ý tưởng nền tảng / trực giác

Thử tay `"crcoakroak"` (chỉ số 0..9). Ta đi từ trái sang phải, mỗi chữ phải giao cho một con ếch:

| i | chữ | ếch nào nhận? | trạng thái các ếch (chữ vừa kêu) |
|---|---|---|---|
| 0 | c | chưa có ếch ⇒ tạo ếch 1 | E1:c |
| 1 | r | E1 đang chờ `r` | E1:r |
| 2 | c | không ai chờ `c` (không ai vừa xong `k`) ⇒ tạo ếch 2 | E1:r, E2:c |
| 3 | o | E1 chờ `o` | E1:o, E2:c |
| 4 | a | E1 | E1:a, E2:c |
| 5 | k | E1 — xong 1 tiếng, rảnh | E1:k, E2:c |
| 6 | r | E2 | E1:k, E2:r |
| 7 | o | E2 | E2:o |
| 8 | a | E2 | E2:a |
| 9 | k | E2 — xong | E1:k, E2:k |

Hết chuỗi, mọi ếch ở trạng thái `k` (đã kêu xong) ⇒ hợp lệ, cần **2** ếch.

Quan sát: ta **không cần biết ếch nào là ếch nào**, chỉ cần biết **có bao nhiêu** ếch ở mỗi trạng thái. Và khi gặp `c`, nếu có ếch đang rảnh (trạng thái `k`) thì dùng lại nó thay vì tạo mới.

In [6]:
# kiểm tra điều kiện cần: số lượng 5 chữ bằng nhau
for s, _ in examples:
    c = Counter(s)
    print(s, [c[ch] for ch in "croak"], "bằng nhau?" , len({c[ch] for ch in "croak"}) == 1)

croakcroak [2, 2, 2, 2, 2] bằng nhau? True
crcoakroak [2, 2, 2, 2, 2] bằng nhau? True
croakcrook [2, 2, 3, 1, 2] bằng nhau? False


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng con ếch

**Ý tưởng:** giữ danh sách `frogs`, mỗi phần tử là chỉ số chữ mà con ếch đó **vừa** kêu (hoặc `4` nếu đang rảnh sau `k`). Với mỗi chữ `ch` có chỉ số `i`:
- nếu `i == 0` (`c`): tìm một ếch rảnh (trạng thái 4) để dùng lại; không có thì **tạo ếch mới**;
- ngược lại: tìm **một ếch bất kỳ** đang ở trạng thái `i-1`; không có ⇒ `-1`.

Cuối cùng mọi ếch phải ở trạng thái 4. Đáp án = số ếch đã tạo.

In [7]:
def frogs_simulate(s):
    frogs = []                         # frogs[j] = chỉ số chữ ếch j vừa kêu
    for ch in s:
        i = "croak".index(ch)
        want = 4 if i == 0 else i - 1
        for j, st in enumerate(frogs):  # quét tuyến tính tìm ếch phù hợp
            if st == want:
                frogs[j] = i
                break
        else:                           # không tìm thấy
            if i == 0:
                frogs.append(0)         # tạo ếch mới
            else:
                return -1
    return len(frogs) if all(st == 4 for st in frogs) else -1

for s, want in examples:
    print(s, frogs_simulate(s), "✔" if frogs_simulate(s) == want else "✘")

croakcroak 1 ✔
crcoakroak 2 ✔
croakcrook -1 ✔


**Độ phức tạp:** mỗi chữ quét tối đa `F` con ếch ⇒ `O(n · F)`, với `F ≤ n/5`. Trường hợp xấu `"ccc...c rrr...r ..."` (`n/5` ếch kêu song song) ⇒ `O(n²/5)` ≈ 4·10^8 bước khi `n = 10^5` — quá chậm trong Python.

**Nút thắt:** ta chỉ cần biết **có** ếch ở trạng thái `want` hay không, không cần biết **ếch nào** ⇒ thay danh sách bằng **bộ đếm theo trạng thái**.

In [8]:
worst = "c" * 2000 + "r" * 2000 + "o" * 2000 + "a" * 2000 + "k" * 2000
t = time.perf_counter(); r = frogs_simulate(worst); el = time.perf_counter() - t
print(f"n={len(worst)}, kết quả {r}, thời gian {el:.3f}s  (n=10^5 sẽ chậm gấp ~100 lần)")

n=10000, kết quả 2000, thời gian 0.222s  (n=10^5 sẽ chậm gấp ~100 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — các ếch cùng trạng thái là "hoán đổi được":** nếu hai ếch đều vừa kêu `r`, giao chữ `o` kế tiếp cho ếch nào cũng như nhau ⇒ chỉ cần đếm `cnt[i]`.

**Insight 2 — tham lam dùng lại ếch rảnh:** khi gặp `c`, **nếu** có ếch đã kêu xong (`cnt[4] > 0`) **thì** dùng lại nó (`cnt[4] -= 1`); **ngược lại** mới tạo ếch mới (`ans += 1`). Dùng lại không bao giờ làm hại: con ếch rảnh đó nếu không dùng thì cũng ngồi không.

**Insight 3 — quy tắc hợp lệ:** với chữ thứ `i > 0`, **nếu** `cnt[i-1] == 0` **thì** không ai có thể kêu chữ này ⇒ `-1`. Hết chuỗi, **nếu** còn ếch ở trạng thái `0..3` **thì** có tiếng kêu dở ⇒ `-1`.

**Insight 4 — công thức gọn:** số ếch **đang kêu dở** tại một thời điểm = `#c − #k` đã thấy. Số ếch tối thiểu = **giá trị lớn nhất** của đại lượng này. Điều kiện hợp lệ ở mọi tiền tố: `#c ≥ #r ≥ #o ≥ #a ≥ #k`; cuối cùng 5 số bằng nhau.

**Bất biến:** sau khi xử lý tiền tố, `ans` = số ếch đã tạo = `max` số ếch bận đồng thời; `sum(cnt) = ans`.

## 7. Các cách tiếp theo

### Cách 2 — Đếm theo trạng thái `cnt[5]` (mô phỏng không tên)

**Từng đoạn:**
1. `cnt = [0]*5`, `ans = 0`.
2. Chữ `c`: nếu `cnt[4] > 0` thì `cnt[4] -= 1` (dùng lại), không thì `ans += 1`; rồi `cnt[0] += 1`.
3. Chữ thứ `i > 0`: nếu `cnt[i-1] == 0` trả `-1`; không thì chuyển `cnt[i-1] -= 1; cnt[i] += 1`.
4. Cuối: nếu `cnt[0..3]` toàn 0 trả `ans`, không thì `-1`.

In [9]:
def frogs_states(s):
    cnt = [0] * 5
    ans = 0
    for ch in s:
        i = IDX[ch]
        if i == 0:
            if cnt[4]:
                cnt[4] -= 1          # dùng lại ếch rảnh
            else:
                ans += 1             # tạo ếch mới
            cnt[0] += 1
        else:
            if cnt[i - 1] == 0:
                return -1
            cnt[i - 1] -= 1
            cnt[i] += 1
    return ans if sum(cnt[:4]) == 0 else -1

for s, want in examples:
    print(s, frogs_states(s), frogs_states(s) == want)

croakcroak 1 True
crcoakroak 2 True
croakcrook -1 True


In [10]:
class SolutionStates:
    def minNumberOfFrogs(self, croakOfFrogs: str) -> int:
        idx = {ch: i for i, ch in enumerate("croak")}
        cnt = [0] * 5
        ans = 0
        for ch in croakOfFrogs:
            i = idx[ch]
            if i == 0:
                if cnt[4]:
                    cnt[4] -= 1
                else:
                    ans += 1
                cnt[0] += 1
            else:
                if cnt[i - 1] == 0:
                    return -1
                cnt[i - 1] -= 1
                cnt[i] += 1
        return ans if sum(cnt[:4]) == 0 else -1

print([SolutionStates().minNumberOfFrogs(s) for s, _ in examples])

[1, 2, -1]


**Độ phức tạp:** `O(n)` thời gian (mỗi chữ `O(1)`), `O(1)` bộ nhớ (5 bộ đếm).

### Cách 3 — Đếm số lần mỗi chữ + "số ếch đang kêu" = `#c − #k`

**Từng đoạn:**
1. `seen = [0]*5` đếm số lần mỗi chữ đã xuất hiện.
2. Sau khi tăng `seen[i]`, nếu `i > 0` và `seen[i] > seen[i-1]` ⇒ có chữ xuất hiện nhiều hơn chữ đứng trước nó ⇒ `-1`.
3. `active = seen[0] - seen[4]` là số ếch đang kêu dở; `best = max(best, active)`.
4. Cuối: phải có `seen[0] == seen[4]` (mọi tiếng đã xong; kết hợp bước 2 ⇒ 5 số bằng nhau).

In [11]:
def frogs_active(s):
    seen = [0] * 5
    best = 0
    for ch in s:
        i = IDX[ch]
        seen[i] += 1
        if i > 0 and seen[i] > seen[i - 1]:
            return -1
        best = max(best, seen[0] - seen[4])
    return best if seen[0] == seen[4] else -1

for s, want in examples:
    print(s, frogs_active(s), frogs_active(s) == want)

croakcroak 1 True
crcoakroak 2 True
croakcrook -1 True


In [12]:
class SolutionActive:
    def minNumberOfFrogs(self, croakOfFrogs: str) -> int:
        c = r = o = a = k = 0
        best = 0
        for ch in croakOfFrogs:
            if ch == "c":
                c += 1
            elif ch == "r":
                r += 1
            elif ch == "o":
                o += 1
            elif ch == "a":
                a += 1
            else:
                k += 1
            if not (c >= r >= o >= a >= k):
                return -1
            best = max(best, c - k)
        return best if c == k else -1

print([SolutionActive().minNumberOfFrogs(s) for s, _ in examples])

[1, 2, -1]


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ. Đây là cách ngắn nhất để viết khi phỏng vấn.

### Cách 4 — Ghép tiếng kêu thành đoạn `[c, k]` rồi quét sự kiện

**Ý tưởng:** ghép `c` thứ `j` với `r` thứ `j`, `o` thứ `j`, `a` thứ `j`, `k` thứ `j` (thứ tự FIFO). Tiếng kêu `j` hợp lệ khi vị trí 5 chữ của nó tăng dần. Mỗi tiếng là một đoạn thời gian `[pos_c, pos_k]`; một con ếch không thể kêu hai đoạn chồng nhau ⇒ cần ít nhất (số đoạn chồng nhau tối đa) con ếch, và số đó là đủ (gán tham lam như xếp phòng họp).

Cách này **minh hoạ** mối liên hệ với bài "Meeting Rooms II" (LeetCode 253).

In [13]:
def frogs_intervals(s):
    pos = [[] for _ in range(5)]
    for t, ch in enumerate(s):
        pos[IDX[ch]].append(t)
    m = len(pos[0])
    if any(len(p) != m for p in pos):
        return -1
    segs = []
    for j in range(m):
        ts = [pos[i][j] for i in range(5)]
        if any(ts[i] > ts[i + 1] for i in range(4)):
            return -1
        segs.append((ts[0], ts[4]))
    events = sorted([(l, 1) for l, r in segs] + [(r + 1, -1) for l, r in segs])
    cur = best = 0
    for _, d in events:
        cur += d
        best = max(best, cur)
    return best

for s, want in examples:
    print(s, frogs_intervals(s), frogs_intervals(s) == want)

croakcroak 1 True
crcoakroak 2 True
croakcrook -1 True


**Vì sao ghép FIFO là đúng?** Nếu có cách hợp lệ nào đó, ta luôn có thể "đổi chéo" để chữ `r` sớm nhất đi với chữ `c` sớm nhất mà vẫn hợp lệ (lập luận trao đổi). Do đó kiểm tra hợp lệ bằng ghép FIFO là đủ. Lưu ý sự kiện đóng đặt ở `r + 1` vì ếch kêu `k` tại thời điểm `r` rồi mới rảnh từ `r + 1`; nhưng vì `c` và `k` không bao giờ cùng vị trí nên dùng `r` hay `r+1` đều cho cùng kết quả ở đây.

**Độ phức tạp:** `O(n log n)` do sort sự kiện (có thể `O(n)` vì vị trí đã có thứ tự), `O(n)` bộ nhớ.

In [14]:
class SolutionIntervals:
    def minNumberOfFrogs(self, croakOfFrogs: str) -> int:
        return frogs_intervals(croakOfFrogs)

print([SolutionIntervals().minNumberOfFrogs(s) for s, _ in examples])

[1, 2, -1]


## 8. TRACE từng bước

Trace cách 2 với `"crcoakroak"`:

| i | chữ | hành động | `cnt` (c,r,o,a,k) | `ans` |
|---|---|---|---|---|
| 0 | c | không ai rảnh ⇒ ếch mới | 1,0,0,0,0 | 1 |
| 1 | r | c→r | 0,1,0,0,0 | 1 |
| 2 | c | không ai rảnh ⇒ ếch mới | 1,1,0,0,0 | 2 |
| 3 | o | r→o | 1,0,1,0,0 | 2 |
| 4 | a | o→a | 1,0,0,1,0 | 2 |
| 5 | k | a→k | 1,0,0,0,1 | 2 |
| 6 | r | c→r | 0,1,0,0,1 | 2 |
| 7 | o | r→o | 0,0,1,0,1 | 2 |
| 8 | a | o→a | 0,0,0,1,1 | 2 |
| 9 | k | a→k | 0,0,0,0,2 | 2 |

In [15]:
def trace_states(s):
    cnt = [0] * 5
    ans = 0
    print(f"chuỗi {s!r}")
    for t, ch in enumerate(s):
        i = IDX[ch]
        if i == 0:
            if cnt[4]:
                cnt[4] -= 1; act = "dùng lại ếch rảnh"
            else:
                ans += 1; act = "TẠO ếch mới"
            cnt[0] += 1
        else:
            if cnt[i - 1] == 0:
                print(f"  t={t:2d} '{ch}': không có ếch nào vừa kêu '{'croak'[i-1]}' -> -1")
                return -1
            cnt[i - 1] -= 1; cnt[i] += 1
            act = f"{'croak'[i-1]}->{ch}"
        print(f"  t={t:2d} '{ch}': {act:18s} cnt={cnt} ans={ans}")
    ok = sum(cnt[:4]) == 0
    print("  cuối: còn tiếng dở?" , not ok, "->", ans if ok else -1)
    return ans if ok else -1

trace_states("crcoakroak")

chuỗi 'crcoakroak'
  t= 0 'c': TẠO ếch mới        cnt=[1, 0, 0, 0, 0] ans=1
  t= 1 'r': c->r               cnt=[0, 1, 0, 0, 0] ans=1
  t= 2 'c': TẠO ếch mới        cnt=[1, 1, 0, 0, 0] ans=2
  t= 3 'o': r->o               cnt=[1, 0, 1, 0, 0] ans=2
  t= 4 'a': o->a               cnt=[1, 0, 0, 1, 0] ans=2
  t= 5 'k': a->k               cnt=[1, 0, 0, 0, 1] ans=2
  t= 6 'r': c->r               cnt=[0, 1, 0, 0, 1] ans=2
  t= 7 'o': r->o               cnt=[0, 0, 1, 0, 1] ans=2
  t= 8 'a': o->a               cnt=[0, 0, 0, 1, 1] ans=2
  t= 9 'k': a->k               cnt=[0, 0, 0, 0, 2] ans=2
  cuối: còn tiếng dở? False -> 2


2

In [16]:
trace_states("croakcrook")
print()
trace_states("croakcr")

chuỗi 'croakcrook'
  t= 0 'c': TẠO ếch mới        cnt=[1, 0, 0, 0, 0] ans=1
  t= 1 'r': c->r               cnt=[0, 1, 0, 0, 0] ans=1
  t= 2 'o': r->o               cnt=[0, 0, 1, 0, 0] ans=1
  t= 3 'a': o->a               cnt=[0, 0, 0, 1, 0] ans=1
  t= 4 'k': a->k               cnt=[0, 0, 0, 0, 1] ans=1
  t= 5 'c': dùng lại ếch rảnh  cnt=[1, 0, 0, 0, 0] ans=1
  t= 6 'r': c->r               cnt=[0, 1, 0, 0, 0] ans=1
  t= 7 'o': r->o               cnt=[0, 0, 1, 0, 0] ans=1
  t= 8 'o': không có ếch nào vừa kêu 'r' -> -1

chuỗi 'croakcr'
  t= 0 'c': TẠO ếch mới        cnt=[1, 0, 0, 0, 0] ans=1
  t= 1 'r': c->r               cnt=[0, 1, 0, 0, 0] ans=1
  t= 2 'o': r->o               cnt=[0, 0, 1, 0, 0] ans=1
  t= 3 'a': o->a               cnt=[0, 0, 0, 1, 0] ans=1
  t= 4 'k': a->k               cnt=[0, 0, 0, 0, 1] ans=1
  t= 5 'c': dùng lại ếch rảnh  cnt=[1, 0, 0, 0, 0] ans=1
  t= 6 'r': c->r               cnt=[0, 1, 0, 0, 0] ans=1
  cuối: còn tiếng dở? True -> -1


-1

Trace cách 3 (số ếch đang kêu) với `"ccrrooaakkcroak"`:

In [17]:
def trace_active(s):
    seen = [0] * 5
    best = 0
    for t, ch in enumerate(s):
        seen[IDX[ch]] += 1
        active = seen[0] - seen[4]
        best = max(best, active)
        print(f"t={t:2d} '{ch}' seen(c,r,o,a,k)={seen} đang kêu={active} best={best}")
    return best

trace_active("ccrrooaakkcroak")

t= 0 'c' seen(c,r,o,a,k)=[1, 0, 0, 0, 0] đang kêu=1 best=1
t= 1 'c' seen(c,r,o,a,k)=[2, 0, 0, 0, 0] đang kêu=2 best=2
t= 2 'r' seen(c,r,o,a,k)=[2, 1, 0, 0, 0] đang kêu=2 best=2
t= 3 'r' seen(c,r,o,a,k)=[2, 2, 0, 0, 0] đang kêu=2 best=2
t= 4 'o' seen(c,r,o,a,k)=[2, 2, 1, 0, 0] đang kêu=2 best=2
t= 5 'o' seen(c,r,o,a,k)=[2, 2, 2, 0, 0] đang kêu=2 best=2
t= 6 'a' seen(c,r,o,a,k)=[2, 2, 2, 1, 0] đang kêu=2 best=2
t= 7 'a' seen(c,r,o,a,k)=[2, 2, 2, 2, 0] đang kêu=2 best=2
t= 8 'k' seen(c,r,o,a,k)=[2, 2, 2, 2, 1] đang kêu=1 best=2
t= 9 'k' seen(c,r,o,a,k)=[2, 2, 2, 2, 2] đang kêu=0 best=2
t=10 'c' seen(c,r,o,a,k)=[3, 2, 2, 2, 2] đang kêu=1 best=2
t=11 'r' seen(c,r,o,a,k)=[3, 3, 2, 2, 2] đang kêu=1 best=2
t=12 'o' seen(c,r,o,a,k)=[3, 3, 3, 2, 2] đang kêu=1 best=2
t=13 'a' seen(c,r,o,a,k)=[3, 3, 3, 3, 2] đang kêu=1 best=2
t=14 'k' seen(c,r,o,a,k)=[3, 3, 3, 3, 3] đang kêu=0 best=2


2

## 9. Code nộp LeetCode

Dùng cách 2 (đếm trạng thái) — `O(n)` / `O(1)`, mô phỏng trực quan:

In [18]:
class Solution:
    def minNumberOfFrogs(self, croakOfFrogs: str) -> int:
        idx = {ch: i for i, ch in enumerate("croak")}
        cnt = [0] * 5          # cnt[i] = số ếch vừa kêu chữ thứ i
        ans = 0                # số ếch đã phải tạo
        for ch in croakOfFrogs:
            i = idx[ch]
            if i == 0:
                if cnt[4]:
                    cnt[4] -= 1          # có ếch rảnh -> dùng lại
                else:
                    ans += 1             # phải thêm ếch
                cnt[0] += 1
            else:
                if cnt[i - 1] == 0:      # không ai chờ chữ này
                    return -1
                cnt[i - 1] -= 1
                cnt[i] += 1
        return ans if sum(cnt[:4]) == 0 else -1

for s, want in examples:
    assert Solution().minNumberOfFrogs(s) == want
print("3 ví dụ: OK")

3 ví dụ: OK


## 10. Vì sao thuật toán đúng

**Hợp lệ ⇔ thuật toán không trả `-1`:**
- Nếu thuật toán chạy hết mà `cnt[0..3] = 0`, nó đã thực sự dựng được một cách gán mỗi chữ cho một ếch, mỗi ếch kêu các tiếng `croak` trọn vẹn ⇒ hợp lệ.
- Nếu tại chữ thứ `i > 0` mà `cnt[i-1] = 0`: trong tiền tố, số chữ `i` vượt số chữ `i-1` (vì mỗi chữ `i` cần một chữ `i-1` đứng trước của cùng ếch) ⇒ không cách gán nào tồn tại. Nếu cuối còn ếch ở trạng thái `0..3` ⇒ số `c` khác số `k` ⇒ có tiếng dở.

**Tối thiểu:** tại mọi thời điểm, số ếch **đang kêu dở** là `#c − #k` (mỗi tiếng đang mở có đúng một `c` chưa có `k` tương ứng). Những ếch đó phải là các con khác nhau ⇒ cần ít nhất `max(#c − #k)` ếch. Thuật toán chỉ tạo ếch mới khi `cnt[4] = 0`, tức mọi ếch hiện có đều đang bận ⇒ sau khi tạo, số ếch = số ếch bận = `#c − #k` hiện tại. Vậy `ans = max(#c − #k)` — đạt đúng cận dưới.

**Bất biến:** `cnt[0] + cnt[1] + cnt[2] + cnt[3] = #c − #k` và `sum(cnt) = ans`.

## 11. Phân tích Time / Space

Gọi `n = len(croakOfFrogs)`, `F` = số ếch cần (`≤ n / 5`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng từng ếch | `O(n · F)` ⇒ worst `O(n²)` | `O(F)` | quét danh sách ếch |
| 2. Đếm trạng thái | `O(n)` | `O(1)` | 5 bộ đếm |
| 3. `#c − #k` | `O(n)` | `O(1)` | ngắn nhất |
| 4. Đoạn + quét | `O(n log n)` | `O(n)` | minh hoạ Meeting Rooms |

In [19]:
rng = random.Random(1)
def gen_valid(rng, frogs, rounds):
    """Sinh chuỗi hợp lệ bằng cách cho 'frogs' con ếch kêu xen kẽ ngẫu nhiên."""
    progress = [[0, rounds] for _ in range(frogs)]   # [chữ kế, số tiếng còn lại]
    out = []
    alive = list(range(frogs))
    while alive:
        j = rng.choice(alive)
        out.append("croak"[progress[j][0]])
        progress[j][0] += 1
        if progress[j][0] == 5:
            progress[j][0] = 0
            progress[j][1] -= 1
            if progress[j][1] == 0:
                alive.remove(j)
    return "".join(out)

big = gen_valid(rng, 200, 100)          # n = 10^5
for name, f in [("states", frogs_states), ("active", frogs_active), ("intervals", frogs_intervals)]:
    t = time.perf_counter(); r = f(big); el = time.perf_counter() - t
    print(f"{name:9s}: n={len(big)}, ếch tối thiểu={r}, {el:.3f}s")

states   : n=100000, ếch tối thiểu=179, 0.010s
active   : n=100000, ếch tối thiểu=179, 0.012s
intervals: n=100000, ếch tối thiểu=179, 0.033s


## 12. Bộ test

1. Ví dụ đề.
2. Edge: `"c"`, `"k"`, `"croak"`, chuỗi dở `"croa"`, đủ số lượng nhưng sai thứ tự `"rcoak"`, `"croakk"`, song song hoàn toàn `"ccrrooaakk"`.
3. **Brute force chính xác** (vét cạn mọi cách gán chữ cho ếch, n nhỏ) để kiểm chứng tham lam là tối ưu.
4. Ngẫu nhiên: chuỗi hợp lệ sinh từ mô phỏng (số ếch cho trước ⇒ đáp án ≤ số đó) và chuỗi ngẫu nhiên bất kỳ (phần lớn `-1`).

In [20]:
impls = {
    "simulate": frogs_simulate, "states": frogs_states, "active": frogs_active, "intervals": frogs_intervals,
    "Solution": lambda s: Solution().minNumberOfFrogs(s),
    "SolActive": lambda s: SolutionActive().minNumberOfFrogs(s),
    "SolStates": lambda s: SolutionStates().minNumberOfFrogs(s),
}
edge = [("c", -1), ("k", -1), ("croak", 1), ("croa", -1), ("rcoak", -1), ("croakk", -1),
        ("ccrrooaakk", 2), ("cccrrroooaaakkk", 3), ("croakcroa", -1), ("kcroa", -1)]
for name, f in impls.items():
    for s, want in examples + edge:
        assert f(s) == want, (name, s, f(s))
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [21]:
def frogs_exact(s, max_frogs=4):
    """Vét cạn: thử số ếch F = 1..max_frogs, gán từng chữ cho một ếch có thể kêu nó (backtracking)."""
    def can(F):
        @lru_cache(maxsize=None)
        def go(t, states):
            if t == len(s):
                return all(st == 4 for st in states)
            i = IDX[s[t]]
            want = 4 if i == 0 else i - 1
            for j in range(F):
                if states[j] == want:
                    ns = states[:j] + (i,) + states[j + 1:]
                    if go(t + 1, tuple(sorted(ns))):
                        return True
            return False
        return go(0, tuple([4] * F))
    for F in range(1, max_frogs + 1):
        if can(F):
            return F
    return -1

rng = random.Random(1419)
checked = 0
for _ in range(250):
    if rng.random() < 0.6:
        s = gen_valid(rng, rng.randint(1, 3), rng.randint(1, 2))
    else:
        s = "".join(rng.choice("croak") for _ in range(rng.choice([5, 10])))
    want = frogs_exact(s)
    for name, f in impls.items():
        assert f(s) == want, (name, s)
    checked += 1
print("đối chiếu với vét cạn:", checked, "chuỗi: OK")

đối chiếu với vét cạn: 250 chuỗi: OK


In [22]:
rng = random.Random(7)
for _ in range(300):
    F = rng.randint(1, 30)
    s = gen_valid(rng, F, rng.randint(1, 5))
    r = Solution().minNumberOfFrogs(s)
    assert 1 <= r <= F
    assert all(f(s) == r for f in impls.values())
    # phá hỏng một ký tự -> thường không hợp lệ; các cách phải đồng ý với nhau
    t = rng.randrange(len(s)); bad = s[:t] + rng.choice("croak") + s[t + 1:]
    assert len({f(bad) for f in impls.values()}) == 1
print("random chuỗi hợp lệ + chuỗi bị phá: OK")

random chuỗi hợp lệ + chuỗi bị phá: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra số lượng các chữ bằng nhau ở cuối, bỏ qua thứ tự

In [23]:
def wrong_only_counts(s):
    c = Counter(s)
    if len({c[ch] for ch in "croak"}) != 1:
        return -1
    best = cur = 0
    for ch in s:
        cur += (ch == "c") - (ch == "k")
        best = max(best, cur)
    return best

for s in ["rcoak", "kcroa", "croka"]:
    print(f"{s!r}: sai -> {wrong_only_counts(s)}, đúng -> {Solution().minNumberOfFrogs(s)}")

'rcoak': sai -> 1, đúng -> -1
'kcroa': sai -> 0, đúng -> -1
'croka': sai -> 1, đúng -> -1


**Vì sao sai:** `"rcoak"` có đủ 1 chữ mỗi loại nhưng `r` xuất hiện trước `c` — không ếch nào kêu được. Số lượng bằng nhau chỉ là **điều kiện cần**.

**Sửa:** kiểm tra ở **mọi tiền tố** `#c ≥ #r ≥ #o ≥ #a ≥ #k` (cách 3) hoặc `cnt[i-1] > 0` trước khi chuyển (cách 2).

### Lỗi 2 — Trả về số chữ `c` (số tiếng kêu) thay vì số ếch đồng thời

In [24]:
def wrong_count_c(s):
    r = Solution().minNumberOfFrogs(s)
    return -1 if r == -1 else s.count("c")      # SAI: ếch có thể kêu nhiều lần

for s in ["croakcroak", "croakcroakcroak", "crcoakroak"]:
    print(f"{s!r}: sai -> {wrong_count_c(s)}, đúng -> {Solution().minNumberOfFrogs(s)}")

'croakcroak': sai -> 2, đúng -> 1
'croakcroakcroak': sai -> 3, đúng -> 1
'crcoakroak': sai -> 2, đúng -> 2


**Vì sao sai:** một con ếch kêu xong có thể kêu tiếp. `"croakcroak"` chỉ cần 1 ếch, không phải 2.

**Sửa:** đáp án là **số tiếng kêu chồng nhau tối đa** = `max(#c − #k)`, hoặc dùng lại ếch rảnh (`cnt[4]`) khi gặp `c`.

### Lỗi 3 — Quên kiểm tra tiếng kêu dở dang ở cuối

In [25]:
def wrong_no_final_check(s):
    cnt = [0] * 5; ans = 0
    for ch in s:
        i = IDX[ch]
        if i == 0:
            if cnt[4]: cnt[4] -= 1
            else: ans += 1
            cnt[0] += 1
        else:
            if cnt[i - 1] == 0: return -1
            cnt[i - 1] -= 1; cnt[i] += 1
    return ans                                    # SAI: thiếu kiểm tra cnt[0..3]

for s in ["croakcr", "c", "croa"]:
    print(f"{s!r}: sai -> {wrong_no_final_check(s)}, đúng -> {Solution().minNumberOfFrogs(s)}")

'croakcr': sai -> 1, đúng -> -1
'c': sai -> 1, đúng -> -1
'croa': sai -> 1, đúng -> -1


**Vì sao sai:** đề yêu cầu mỗi ếch phải kêu **đủ 5 chữ**. `"croakcr"` kết thúc khi một ếch mới kêu `cr` ⇒ không hợp lệ.

**Sửa:** cuối vòng lặp, `return ans if sum(cnt[:4]) == 0 else -1` (hoặc `c == k` ở cách 3).

### Lỗi 4 — Không kiểm tra `cnt[i-1] > 0` (bộ đếm âm)

In [26]:
def wrong_negative(s):
    cnt = [0] * 5; ans = 0
    for ch in s:
        i = IDX[ch]
        if i == 0:
            if cnt[4] > 0: cnt[4] -= 1
            else: ans += 1
            cnt[0] += 1
        else:
            cnt[i - 1] -= 1; cnt[i] += 1          # SAI: có thể âm
    return ans if sum(cnt[:4]) == 0 else -1

for s in ["rcoak", "ckroa"]:
    print(f"{s!r}: sai -> {wrong_negative(s)}, đúng -> {Solution().minNumberOfFrogs(s)}")

'rcoak': sai -> 1, đúng -> -1
'ckroa': sai -> 1, đúng -> -1


**Vì sao sai:** với `"rcoak"`, `cnt[0]` xuống `-1` ở chữ `r` rồi được bù lại bởi `c` phía sau, tổng cuối vẫn 0 nên trả `1`. Bộ đếm âm nghĩa là "một ếch kêu `r` trước khi kêu `c`" — vô nghĩa.

**Sửa:** kiểm tra `if cnt[i-1] == 0: return -1` **trước** khi trừ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Nhiều tiến trình xen kẽ**, mỗi tiến trình đi qua dãy bước cố định" ⇒ đếm số tiến trình ở mỗi bước (máy trạng thái bằng mảng đếm).
- "**Số tác nhân tối thiểu**" cho các công việc có thời gian bắt đầu/kết thúc ⇒ **số đoạn chồng nhau tối đa** (Meeting Rooms II, Car Pooling).
- Kiểm tra hợp lệ của chuỗi "trộn" luôn gồm hai phần: **mọi tiền tố** (không vượt) và **cuối cùng** (khép kín) — giống kiểm tra dấu ngoặc.
- Trong Python, `if not (c >= r >= o >= a >= k)` dùng so sánh dây chuyền rất gọn.
- Test nhanh: `"croakcroak"→1`, `"crcoakroak"→2`, `"rcoak"→-1`, `"croakcr"→-1`, `"ccrrooaakk"→2`.

## 15. Biến thể của bài

1. **Từ bất kỳ có các chữ khác nhau** (ví dụ `"meow"`): thay `"croak"` bằng từ đó — thuật toán y hệt.
2. **Từ có chữ lặp** (ví dụ `"hello"`): một chữ ứng với nhiều trạng thái ⇒ phải chọn trạng thái nào (thường chọn trạng thái "xa nhất" một cách tham lam, cần cẩn thận).
3. **Trả về cách gán** chữ cho từng ếch (mỗi chữ thuộc ếch số mấy).
4. **Kiểm tra dấu ngoặc** (`"()"` là từ dài 2) — LeetCode 20, 1541 là trường hợp riêng.
5. **Meeting Rooms II** (253) / **Car Pooling** (1094): cùng ý tưởng đếm chồng nhau.

Code biến thể 1:

In [27]:
def min_animals(s, word):
    pos = {ch: i for i, ch in enumerate(word)}
    L = len(word)
    cnt = [0] * L
    ans = 0
    for ch in s:
        if ch not in pos:
            return -1
        i = pos[ch]
        if i == 0:
            if cnt[L - 1]: cnt[L - 1] -= 1
            else: ans += 1
            cnt[0] += 1
        else:
            if cnt[i - 1] == 0: return -1
            cnt[i - 1] -= 1; cnt[i] += 1
    return ans if sum(cnt[:L - 1]) == 0 else -1

print(min_animals("mmeoewow", "meow"), min_animals("meowmeow", "meow"), min_animals("crcoakroak", "croak"))

2 1 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng ếch | Danh sách trạng thái từng ếch, quét tìm ếch phù hợp | `O(n·F)` | `O(F)` | Hiểu bài, cần biết ếch nào kêu chữ nào |
| 2. Đếm trạng thái | `cnt[5]`, dùng lại ếch rảnh khi gặp `c` | `O(n)` | `O(1)` | Mặc định, dễ mở rộng cho từ khác |
| 3. `#c − #k` | Kiểm tra dây chuyền `c≥r≥o≥a≥k`, max số ếch đang kêu | `O(n)` | `O(1)` | Code ngắn nhất |
| 4. Đoạn + quét | Ghép FIFO thành đoạn `[c,k]`, đếm chồng nhau | `O(n log n)` | `O(n)` | Liên hệ Meeting Rooms, chứng minh |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().minNumberOfFrogs("ccroakroak")` trả về gì? Còn `"croakcroa"`?

### Bài 2 (tính tay)
Chạy tay cách 2 trên `"crocakcroraoakk"`: ghi `cnt` và `ans` sau mỗi chữ. Đáp án là bao nhiêu?

### Bài 3 (điền TODO)
Hoàn thiện `is_valid_croak(s)` chỉ trả `True/False` (chuỗi có phải sự trộn lẫn hợp lệ không), dùng cách đếm dây chuyền.

### Bài 4 (tự viết lại)
Viết `max_active_moment(s)`: trả về **chỉ số đầu tiên** `t` mà tại đó số ếch đang kêu đạt cực đại (giả sử `s` hợp lệ).

### Bài 5 (mini challenge)
Viết `assign_frogs(s)`: trả về list `owner` cùng độ dài `s`, `owner[t]` là số hiệu ếch (0, 1, 2…) kêu chữ `s[t]`, dùng **ít ếch nhất**; trả `None` nếu không hợp lệ.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def is_valid_croak(s):
    seen = [0] * 5
    for ch in s:
        # TODO: tăng seen, kiểm tra seen[i] <= seen[i-1]
        pass
    # TODO: kiểm tra cuối seen[0] == seen[4]
    pass

# Bài 4 — TODO
def max_active_moment(s):
    # TODO: duyệt, tính active = #c - #k, nhớ t đầu tiên đạt max
    pass

# Bài 5 — TODO
def assign_frogs(s):
    # TODO: waiting[i] = list các ếch vừa kêu chữ i (dùng như stack/queue)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `"ccroakroak"`: hai chữ `c` liền nhau ⇒ cần 2 ếch bận cùng lúc. (2) Mọi chữ sau đều có ếch chờ. (3) Đáp án `2`. `"croakcroa"` dở ở cuối ⇒ `-1`.

**Bài 2.** (1) Chữ `c` đầu tạo ếch 1; chữ `c` ở vị trí 3 phải tạo ếch 2 (chưa ai xong). (2) Ếch 1 xong ở vị trí 5 (`k`), chữ `c` ở vị trí 6 dùng lại ếch 1. (3) Cuối `cnt = [0,0,0,0,2]`, đáp án `2`.

**Bài 3.** (1) `i = IDX[ch]; seen[i] += 1`. (2) `if i and seen[i] > seen[i-1]: return False`. (3) `return seen[0] == seen[4]`.

**Bài 4.** (1) `active = c - k`. (2) Chỉ cập nhật khi `active > best` (dấu `>` để giữ lần đầu). (3) Trả `t`.

**Bài 5.** (1) `waiting = [[] for _ in range(5)]`, `waiting[4]` là ếch rảnh. (2) Chữ `c`: lấy ếch rảnh nếu có, không thì cấp số mới `len(...)`; chữ `i>0`: lấy một ếch từ `waiting[i-1]`. (3) Đẩy ếch đó vào `waiting[i]`; cuối kiểm tra `waiting[0..3]` rỗng.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert Solution().minNumberOfFrogs("ccroakroak") == 2
assert Solution().minNumberOfFrogs("croakcroa") == -1
# Bài 2
assert Solution().minNumberOfFrogs("crocakcroraoakk") == 2

# Bài 3
def is_valid_croak_sol(s):
    seen = [0] * 5
    for ch in s:
        i = IDX[ch]
        seen[i] += 1
        if i and seen[i] > seen[i - 1]:
            return False
    return seen[0] == seen[4]

# Bài 4
def max_active_moment_sol(s):
    c = k = 0
    best, at = -1, -1
    for t, ch in enumerate(s):
        c += ch == "c"
        k += ch == "k"
        if c - k > best:
            best, at = c - k, t
    return at

assert max_active_moment_sol("crcoakroak") == 2
assert max_active_moment_sol("croakcroak") == 0

# Bài 5
def assign_frogs_sol(s):
    waiting = [[] for _ in range(5)]
    owner = []
    nxt = 0
    for ch in s:
        i = IDX[ch]
        src = 4 if i == 0 else i - 1
        if waiting[src]:
            f = waiting[src].pop()
        elif i == 0:
            f = nxt; nxt += 1
        else:
            return None
        owner.append(f)
        waiting[i].append(f)
    if any(waiting[i] for i in range(4)):
        return None
    return owner

def check_assignment(s, owner):
    by = {}
    for ch, f in zip(s, owner):
        by.setdefault(f, []).append(ch)
    return all("".join(v) == "croak" * (len(v) // 5) and len(v) % 5 == 0 for v in by.values())

rng = random.Random(5)
for _ in range(300):
    if rng.random() < 0.7:
        s = gen_valid(rng, rng.randint(1, 6), rng.randint(1, 3))
    else:
        s = "".join(rng.choice("croak") for _ in range(rng.randint(1, 15)))
    r = Solution().minNumberOfFrogs(s)
    assert is_valid_croak_sol(s) == (r != -1)
    ow = assign_frogs_sol(s)
    if r == -1:
        assert ow is None
    else:
        assert ow is not None and check_assignment(s, ow) and len(set(ow)) == r
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template máy trạng thái đếm (từ có chữ khác nhau):**

```python
idx = {ch: i for i, ch in enumerate(WORD)}
L = len(WORD); cnt = [0] * L; ans = 0
for ch in s:
    i = idx[ch]
    if i == 0:
        if cnt[L-1]: cnt[L-1] -= 1      # dùng lại tác nhân rảnh
        else: ans += 1                  # thêm tác nhân
        cnt[0] += 1
    else:
        if cnt[i-1] == 0: return -1     # sai thứ tự
        cnt[i-1] -= 1; cnt[i] += 1
return ans if sum(cnt[:L-1]) == 0 else -1   # còn dở?
```

**Template ngắn (đếm dây chuyền):** `c ≥ r ≥ o ≥ a ≥ k` ở mọi tiền tố, `best = max(best, c - k)`, cuối `c == k`.

**Checklist:**
- [ ] Kiểm tra thứ tự ở **mọi tiền tố**, không chỉ đếm cuối.
- [ ] Dùng lại ếch đã kêu xong khi gặp `c` (không trả số chữ `c`).
- [ ] Kiểm tra tiếng kêu dở ở cuối ⇒ `-1`.
- [ ] Không để bộ đếm âm.
- [ ] Đã thử `"croakcroak"`, `"crcoakroak"`, `"rcoak"`, `"croakcr"`, `"c"`.